# Lesson 4: Python on GPUs and HEP examples



Adapted from the upstream [array-oriented-programming GPU lesson](https://hsf-training.github.io/array-oriented-programming/5-gpu.html) and reorganized for this course session.



Use this notebook as the guided walkthrough. Then use `lesson-4-workbook.ipynb` for practice and `lesson-4-project.ipynb` for the longer exercise set.



Session flow: verify the CUDA Python environment, compare NumPy and CuPy, write explicit GPU kernels with CuPy and Numba, move a HEP example to the GPU with Awkward Array, then continue to the project notebook.



Note: some figure assets from the upstream training are not bundled in this checkout, and the HEP examples expect their datasets in the local `04/data/` directory.

Slides: [Open the Session 3 presentation](https://1drv.ms/p/c/5a70ac10b7f66de0/IQRJghX87dh-SaFOcgFho8kWAT6QY4Ukeox9B9u2-dS1zeI?em=2&wdAr=1.7777777777777777).



The previous iframe embed was removed because it renders unreliably in notebook viewers and often looked broken when the external Office viewer was blocked.

GPUs are a natural fit for array-oriented programming: they are optimized for applying the same operation to many data elements at once, while CPU-style pointer-heavy and object-heavy patterns map poorly to GPU hardware.



In Python, the practical consequence is that GPU work usually starts with array libraries. CuPy gives you NumPy-like arrays that live on the GPU, and Numba lets you write explicit CUDA kernels in Python when array composition is no longer enough.



This lesson keeps those GPU-specific differences visible rather than hiding them behind a generic `device="cuda"` switch. The goal is to understand where the speedup comes from, what data movement costs, and when you need an explicit kernel rather than a higher-level array expression.

## Required packages

This notebook uses `cupy` and `numba` for GPU work, plus `awkward` and `uproot` for the HEP example.


If your training environment already has these packages, skip the install cell below. Otherwise, install the notebook dependencies here and then reload the kernel.


For this notebook, either of these package-install approaches is fine:

```bash
conda install cuda-version=12 cupy numba awkward uproot
```

```bash
pip install "cupy-cuda12x>=13,<14" "numba>=0.60" "awkward>=2" "uproot>=5"
```

JAX is mentioned later for context, but it is not required for this notebook.

In [ ]:
%pip install "cupy-cuda12x>=13,<14" "numba>=0.60" "awkward>=2" "uproot>=5"

If you ran the install cell, reload the kernel before continuing with the CuPy import test.

In [ ]:
import cupy as cp

In [ ]:
a = cp.random.normal(0, 1, 5 * 1024)
a

In [ ]:
cp.sum(a.reshape(5, 1024), axis=1)

In [ ]:
import numba as nb
import numba.cuda

In [ ]:
@nb.cuda.jit
def zero_out(array):
    i = nb.cuda.grid(1)
    array[i] = 0

In [ ]:
zero_out[1024, 5](a)

In [ ]:
a

## Brief introduction to GPU programming

A GPU is a device attached to a computer (usually separate from the main motherboard, through a PCI bus) that has its own memory and can perform calculations.

_Upstream CPU-versus-GPU comparison figure omitted in this repository checkout._

It differs from the CPUs in several ways. Whereas a typical computer has several CPUs that can all perform calculations independently in parallel, and has a hardware-managed CPU cache to speed up access to recently used data,

* a GPU consists of _thousands_ of independent processing units that can work in parallel,
* with a much slower clock rate than a CPU,
* small groups (32 or 64) of processing units aren't independent; they have to perform the same instructions, but they can apply them to different data,
* memory within the GPU is managed manually by the programmer.

In Seymour Cray's metaphor, "If you were plowing a field, which would you rather use: two strong oxen or 1024 chickens?" a GPU is the thousand chickens. Certain types of problems are faster and more energy efficient to solve with a thousand chickens.

An algorithm intended for a CPU, like this:

```cuda
void some_function(float* array, int index) {
    array[index] = ...;
}

for (int i = 0;  i < 1000000;  i++) {
    some_function(array, i);
}
```

would get translated into something like this for a GPU:

```cuda
__global__ some_function(float* array) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    array[index] = ...;
}

some_function<<<blocks_per_grid, threads_per_block>>>(array);
```

The explicit iteration over elements of an `array` (in sequential order) on the CPU is replaced by a GPU "kernel" function that applies to just one array element, and each kernel-execution "thread" might run at any time. The execution is organized into "blocks": threads within a block run at the same time on shared resources, and the GPU schedules block execution in a way that keeps its streaming multiprocessors busy. The `array` must already be in the GPU's global memory.

_Upstream schematic of GPU blocks in flight omitted in this repository checkout._

The `blocks_per_grid` and `threads_per_block` are units of work-they control how computations are scheduled on the GPU-not units of array allocation. However, for 1-dimensional data, it's often best to define them in terms of the array size:

```cuda
threads_per_block = 1024;   // maximum possible on most GPUs
blocks_per_grid = int(ceil(1.0 * size_of_array / threads_per_block));
```

This minimizes the number of blocks needed to make sure that each array element is updated by exactly one thread.

_Upstream array-to-block mapping figure omitted in this repository checkout._

If the `size_of_array` doesn't fit neatly into an integer number of blocks, a few threads will be wasted. To make sure that they don't update uninitialized data, you usually need to check for an excess in the kernel function:

```cuda
__global__ some_function(float* array) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    if (index < size_of_array) {
        array[index] = ...;
    }
}
```

We'll see this with explicit examples in the next few sections.

## CuPy

[CuPy](https://cupy.dev/) is a library with _mostly_ the same interface as NumPy, but all arrays are in GPU memory, rather than the motherboard's RAM. Converting a NumPy array into a CuPy array, and vice-versa, copies data from RAM to GPU global memory and back.

In [ ]:
import numpy as np
import cupy as cp

In [ ]:
array_in_ram = np.random.uniform(0, 1, 100000000)

In [ ]:
array_on_gpu = cp.asarray(array_in_ram)

In [ ]:
%%timeit -r1 -n10

array_in_ram[:] += 1

In [ ]:
%%timeit -r1 -n10

array_on_gpu[:] += 1

It's important that the distinction is visible like this, since copying an array between RAM and the GPU is often more time-consuming than the computation itself.

In [ ]:
%%timeit -r1 -n10

cp.asarray(array_in_ram)   # from RAM to GPU

In [ ]:
%%timeit -r1 -n10

array_on_gpu.get()         # from GPU to RAM

Thus, you'll want to keep the data on the device that is performing calculations for as long as possible. If data can be created on the GPU, such as random numbers for a Monte Carlo calculation, use CuPy's functions to do it.

In [ ]:
array_on_gpu = cp.random.uniform(0, 1, 100000000)

We can see more of what's going on by running Nvidia's `nsys-ui` profiler on the following line:

In [ ]:
array = cp.random.uniform(5, 10, 100000000)
array.sort()
array

![](img/nsys-profiler-1.png){. width="100%"}

The CUDA kernels (blue) from the first line runs

* `cupy_random_x_mod_1`
* `cupy_scale`

to make the random numbers, followed by

* `DeviceRadixSortHistogramKernel`
* `DeviceRadixSortOnesweepKernel`

to sort the random numbers. Meanwhile on the CPU (green), Python waits for the result with a

* `cudaStreamSynchronize`

Since the GPU is a separate device from the CPU, it runs independently. If you don't ask for the result, the Python function can finish while the GPU calculation is still running. It's only when you want to print the result, copy it to a NumPy array, or get any element as a Python number that CuPy asks the GPU to "synchronize"—wait for computation to be finished.

Since the GPU runs independently, what should it do if it encounters an error part-way through processing? It can't raise a Python error (the Python function call has already finished), so it has to do something else instead. CuPy defines some results that would be errors in NumPy. For instance, this array-slice asks for elements beyond the end of the array:

In [ ]:
array = np.array([0.0, 1.1, 2.2, 3.3, 4.4, 5.5])

array[np.array([2, 3, 5, 6, 7, 8])]

But CuPy returns a result by wrapping the indexes around to the beginning of the array (6 → 0, 7 → 1, 8 → 2):

In [ ]:
array = cp.array([0.0, 1.1, 2.2, 3.3, 4.4, 5.5])

array[cp.array([2, 3, 5, 6, 7, 8])]

_Nsight Systems screenshot omitted in this repository checkout._

The CUDA kernels (blue) from the first line are

* `cupy_random_x_mod_1`
* `cupy_scale`

to make the random numbers, followed by the elementwise arithmetic kernels used in the calculation below.

In [ ]:
a = cp.random.uniform(5, 10, 1000000)
b = cp.random.uniform(10, 20, 1000000)
c = cp.random.uniform(-0.1, 0.1, 1000000)

(-b + cp.sqrt(b**2 - 4*a*c)) / (2*a)

is roughly equivalent to

In [ ]:
tmp1 = cp.negative(b)            # -b
tmp2 = cp.square(b)              # b**2
tmp3 = cp.multiply(4, a)         # 4*a
tmp4 = cp.multiply(tmp3, c)      # tmp3*c
del tmp3
tmp5 = cp.subtract(tmp2, tmp4)   # tmp2 - tmp4
del tmp2, tmp4
tmp6 = cp.sqrt(tmp5)             # sqrt(tmp5)
del tmp5
tmp7 = cp.add(tmp1, tmp6)        # tmp1 + tmp6
del tmp1, tmp6
tmp8 = cp.multiply(2, a)         # 2*a
np.divide(tmp7, tmp8)            # tmp7 / tmp8

Here's what that looks like in the profiler:

![](img/nsys-profiler-2.png){. width="100%"}

Each of the

* `cupy_power__float64_float_float64`
* `cupy_multiply__float_float64_float64`
* `cupy_sqrt__float64_float64`
* `cupy_true_divide__float64_float64_float64`
* `cupy_multiply__float64_float64_float64`
* `cupy_subtract__float64_float64_float64`
* `cupy_add__float64_float64_float64`
* `cupy_negative__float64_float64`

kernels runs quickly, but there are time-gaps between them in which arrays are allocated and dynamic code decides what to do next. It would be faster if the whole quadratic formula were "fused" into a single kernel.

To support this, CuPy has a JIT compiler that lets you write C++ CUDA code. For instance, the worst part of the calculation above is using a general floating-point `power` function,

* `cupy_power__float64_float_float64`

to compute `power(b, 2)`, which should be just `b * b`. Let's define a better function for "raising to an integer power".

In [ ]:
intpow = cp.ElementwiseKernel("float64 x, int64 n", "float64 out", '''
    out = 1.0;
    for (int i = 0;  i < n;  i++) {
        out *= x;
    }
''', "intpow")
intpow

In [ ]:
intpow(b, 2)

In [ ]:
b**2

We can also do this with the whole quadratic formula.

In [ ]:
quadratic_formula = cp.ElementwiseKernel("float64 a, float64 b, float64 c", "float64 out", '''
    out = (-b + sqrt(b*b - 4*a*c)) / (2*a);
''', "quadratic_formula")

quadratic_formula(a, b, c)

In [ ]:
%%timeit -r1 -n1000

(-b + cp.sqrt(b**2 - 4*a*c)) / (2*a)

In [ ]:
%%timeit -r1 -n1000

quadratic_formula(a, b, c)

Both of the above used [ElementwiseKernel](https://docs.cupy.dev/en/stable/user_guide/kernel.html), which is like a NumPy ufunc: they take arrays and apply the function to each element. We could also write a fully generic [RawKernel](https://docs.cupy.dev/en/stable/user_guide/kernel.html#raw-kernels), but then we'd have to manage the `threads_per_block` and `blocks_per_grid` manually.

In [ ]:
quadratic_formula_raw = cp.RawKernel(r'''
extern "C" __global__
void quadratic_formula_raw(const double* a, const double* b, const double* c, int length, double* out) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < length) {
        out[i] = (-b[i] + sqrt(b[i]*b[i] - 4*a[i]*c[i])) / (2*a[i]);
    }
}
''', "quadratic_formula_raw")

out = cp.empty_like(a)

threads_per_block = 1024
blocks_per_grid = int(np.ceil(len(out) / 1024))

quadratic_formula_raw((blocks_per_grid,), (threads_per_block,), (a, b, c, len(out), out))

out

Here's what that looks like in the profiler:

_Nsight Systems screenshot omitted in this repository checkout._

Each of the entries below is a separate GPU kernel launch, which is why a fused custom kernel can be faster than composing many small array operations:

* `cupy_power__float64_float_float64`

## Numba

Just as Numba can JIT compile Python for CPUs, it can [JIT compile Python to CUDA](https://numba.readthedocs.io/en/stable/cuda/index.html).

Here's the quadratic formula as a single CUDA kernel without writing any C++.

In [ ]:
import numba.cuda  # must be explicitly imported
import math        # note that Numba-CUDA requires math.*; you can't use np.*

In [ ]:
@nb.cuda.jit
def quadratic_formula_numba_cuda(a, b, c, out):
    i = nb.cuda.grid(1)   # 1-dimensional
    if i < len(out):
        out[i] = (-b[i] + math.sqrt(b[i]**2 - 4*a[i]*c[i])) / (2*a[i])

out = cp.empty_like(a)

threads_per_block = 1024
blocks_per_grid = int(np.ceil(len(out) / 1024))

quadratic_formula_numba_cuda[blocks_per_grid, threads_per_block](a, b, c, out)

out

Here's what it looks like in the profiler:

![](img/nsys-profiler-3.png){. width="100%"}

The name is a long random string, but it all runs in a short block of time.

Note that the structure of a `@nb.cuda.jit` compiled function is equivalent to a C++ CUDA function: all that has changed is the Python syntax and the names of some of the functions:

```cuda
extern "C" __global__
void quadratic_formula_raw(const double* a, const double* b, const double* c, int length, double* out) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < length) {
        out[i] = (-b[i] + sqrt(b[i]*b[i] - 4*a[i]*c[i])) / (2*a[i]);
    }
}

quadratic_formula_raw<<<blocks_per_grid, threads_per_block>>>(a, b, c, size_of_array, out);
```

versus

```python
@nb.cuda.jit
def quadratic_formula_numba_cuda(a, b, c, out):
    i = nb.cuda.grid(1)   # 1-dimensional
    if i < len(out):
        out[i] = (-b[i] + math.sqrt(b[i]**2 - 4*a[i]*c[i])) / (2*a[i])

quadratic_formula_numba_cuda[blocks_per_grid, threads_per_block](a, b, c, out)
```

You still have to choose an optimal `blocks_per_grid` and `threads_per_block`, and the kernel gets its thread index through a special function, [nb.cuda.grid](https://numba.readthedocs.io/en/stable/cuda-reference/kernel.html#numba.cuda.grid).

## JAX

JAX is worth knowing about because it can also target GPUs and fuse array-oriented operations aggressively. We are not using it directly in this repository notebook, though, because the upstream Mandelbrot companion notebook and comparison figure are not bundled here.



For this session, the practical takeaway is simpler: CuPy gives a NumPy-like GPU array interface, Numba lets you write explicit CUDA kernels in Python, and JAX is another option in the same ecosystem when you want graph-level fusion and automatic differentiation.

## Awkward Array

Awkward Arrays can be copied to a GPU using [ak.to_backend](https://awkward-array.org/doc/main/reference/generated/ak.to_backend.html) with `backend="cuda"`.



In this local course checkout, the Higgs example below reads the ROOT file from the local `04/data/` directory.

In [ ]:
from pathlib import Path


import awkward as ak

import uproot



def resolve_data_root():

    data_entry = Path("data")

    if data_entry.is_dir():

        return data_entry

    if data_entry.is_file():

        pointer_target = Path(data_entry.read_text().strip())

        if pointer_target.exists():

            return pointer_target

        raise FileNotFoundError(

            f"The data pointer file {data_entry} points to {pointer_target}, which does not exist in this checkout."

        )

    raise FileNotFoundError("Expected either a data directory or a pointer file named 'data'.")



DATA_ROOT = resolve_data_root()

In [ ]:
with uproot.open(str(DATA_ROOT / "SMHiggsToZZTo4L.root") + ":Events") as tree:

    events_pt, events_eta, events_phi, events_charge = tree.arrays(

        ["Electron_pt", "Electron_eta", "Electron_phi", "Electron_charge"], how=tuple

    )

In [ ]:
electrons = ak.to_backend(
    ak.zip({
        "pt": events_pt,
        "eta": events_eta,
        "phi": events_phi,
        "charge": events_charge,
    },
    with_name="Momentum4D",
), "cuda")

electrons

Here's what it looks like in the profiler:

_Nsight Systems screenshot omitted in this repository checkout._

The name is a long random string, but it all runs in a short block of time because the Numba-generated CUDA kernel is executing the whole calculation directly.

The structure of a `@nb.cuda.jit` compiled function is close to a C++ CUDA function: all that has changed is the Python syntax and the names of some helper functions.

In [ ]:
e1, e2 = ak.unzip(ak.combinations(electrons, 2))
z_mass = np.sqrt(
    2*e1.pt*e2.pt * (np.cosh(e1.eta - e2.eta) - np.cos(e1.phi - e2.phi))
)
np.max(z_mass, axis=-1)

GPU-bound Awkward Arrays can also be iterated over in Numba-CUDA, which allows for imperative code. However, the output must be a non-ragged CuPy array.

In [ ]:
ak.numba.register_and_check()

@nb.cuda.jit(extensions=[ak.numba.cuda])
def mass_of_heaviest_dielectron(electrons, out):
    thread_idx = nb.cuda.grid(1)
    if thread_idx < len(electrons):
        electrons_in_one_event = electrons[thread_idx]
        for i, e1 in enumerate(electrons_in_one_event):
            for e2 in electrons_in_one_event[i + 1:]:
                if e1.charge != e2.charge:
                    m = math.sqrt(
                        2*e1.pt*e2.pt * (math.cosh(e1.eta - e2.eta) - math.cos(e1.phi - e2.phi))
                    )
                    if m > out[thread_idx]:
                        out[thread_idx] = m

threads_per_block = 1024
blocks_per_grid = int(np.ceil(len(electrons) / 1024))

out = cp.zeros(len(electrons), dtype=np.float32)
mass_of_heaviest_dielectron[blocks_per_grid, threads_per_block](electrons, out)

out

## Lesson 4 project: Histograms and Monte Carlo on GPUs

Continue with `lesson-4-project.ipynb` in this `04` directory for longer exercises on fused kernels, histogramming, and Monte Carlo on the GPU. It uses the same environment assumptions and the same local `04/data/` directory as this walkthrough.